[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/40_linear_regression.ipynb)

# 🟡 Medium: Linear Regression

Implement **linear regression** using three different approaches — all in pure PyTorch.

Given data `X` of shape `(N, D)` and targets `y` of shape `(N,)`, find weight `w` of shape `(D,)` and bias `b` (scalar) such that:

$$\hat{y} = Xw + b$$

### Signature
```python
class LinearRegression:
    def closed_form(self, X: Tensor, y: Tensor) -> tuple[Tensor, Tensor]: ...
    def gradient_descent(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
    def nn_linear(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
```

All methods return `(w, b)` where `w` has shape `(D,)` and `b` has shape `()`.

### Method 1 — Closed-Form (Normal Equation)
Augment X with a ones column, then solve:

$$\theta = (X_{aug}^T X_{aug})^{-1} X_{aug}^T y$$

Or use `torch.linalg.lstsq` / `torch.linalg.solve`.

### Method 2 — Gradient Descent from Scratch
Initialize `w` and `b` to zeros. Repeat for `steps` iterations:
```
pred = X @ w + b
error = pred - y
grad_w = (2/N) * X^T @ error
grad_b = (2/N) * error.sum()
w -= lr * grad_w
b -= lr * grad_b
```

### Method 3 — PyTorch nn.Linear
Create `nn.Linear(D, 1)`, use `nn.MSELoss` and an optimizer (e.g., `torch.optim.SGD`).
After training, extract `w` and `b` from the layer.

### Rules
- All inputs and outputs must be **PyTorch tensors**
- Do **NOT** use numpy or sklearn
- `closed_form` must not use iterative optimization
- `gradient_descent` must manually compute gradients (no `autograd`)
- `nn_linear` should use `torch.nn.Linear` and `loss.backward()`

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn

In [62]:
# ✏️ YOUR IMPLEMENTATION HERE

class LinearRegression:
    def closed_form(self, X: torch.Tensor, y: torch.Tensor):
        """Normal equation: w = (X^T X)^{-1} X^T y"""

        # ones会默认float32，float64或GPU会出问题，使用torch.new_ones((X.shape[0], 1))

        X  = torch.cat((torch.ones([X.shape[0], 1]), X),dim=1)
        xt = torch.transpose(X, 0, 1)
        z = torch.inverse(torch.matmul(xt, X))
        w = torch.matmul(torch.matmul(z, xt), y)

        # torch.linalg.lstsq(X, y).solution 数值上更稳定


        b = w[0]
        w = w[1:]
        return w, b

    def gradient_descent(self, X: torch.Tensor, y: torch.Tensor,
                         lr: float = 0.01, steps: int = 1000):
        """Manual gradient descent loop"""


        """
        w = torch.zeros(X.shape[1], 1)
        b = torch.zeros(1, 1)
        N = X.shape[0]

        for _ in range(steps):
          pred = torch.matmul(X, w) + b
          error = pred - y.unsqueeze(-1)
          grad_w = torch.matmul(X.transpose(0,1), error) * 2 / N
          grad_b = torch.sum(error) * 2 / N
          w -= lr * grad_w
          b -= lr * grad_b

        return w.squeeze(-1), b.squeeze(-1)
        """


        # 直接使用1D而不是2D
        n_samples, n_features = X.shape
        w = X.new_zeros(n_features)
        b = X.new_zeros(())

        for _ in range(steps):
            pred = X @ w + b
            error = pred - y

            grad_w = (2.0 / n_samples) * (X.T @ error)
            grad_b = 2.0 * error.mean()

            w -= lr * grad_w
            b -= lr * grad_b

        return w, b


    def nn_linear(self, X: torch.Tensor, y: torch.Tensor,
                  lr: float = 0.01, steps: int = 1000):
        """Train nn.Linear with autograd"""

        #w = torch.zeros(X.shape[1], requires_grad=True)
        #b = torch.zeros(1, requires_grad=True)

        class MyModel(nn.Module):
            def __init__(self, D):
                super().__init__()
                self.linear = nn.Linear(D, 1)

            def forward(self, x):
                return self.linear(x)

        D = X.shape[1]
        model = MyModel(D)

        # Optional: use the same zero initialization as manual GD,
        # making the two optimization methods easier to compare.
        nn.init.zeros_(model.linear.weight)
        nn.init.zeros_(model.linear.bias)

        optim = torch.optim.SGD(model.parameters(), lr)
        loss_fn = nn.MSELoss()

        y = y.unsqueeze(-1)

        model.train()
        for _ in range(steps):
            optim.zero_grad()
            #pred = model.forward(X)
            pred = model(X)
            loss = loss_fn(pred, y)
            loss.backward()
            optim.step()


        #w = model.state_dict()['linear.weight']
        #b = model.state_dict()['linear.bias']
        with torch.no_grad():
            w = model.linear.weight.squeeze(0).clone()
            b = model.linear.bias.squeeze(0).clone()

        return (w, b)

In [63]:
# 🧪 Debug
torch.manual_seed(42)
X = torch.randn(100, 3)
true_w = torch.tensor([2.0, -1.0, 0.5])
y = X @ true_w + 3.0

model = LinearRegression()

w_cf, b_cf = model.closed_form(X, y)
print(f"Closed-form:  w={w_cf}, b={b_cf.item():.4f}")

w_gd, b_gd = model.gradient_descent(X, y, lr=0.05, steps=2000)
print(f"Grad descent: w={w_gd}, b={b_gd.item():.4f}")

w_nn, b_nn = model.nn_linear(X, y, lr=0.05, steps=2000)
print(f"nn.Linear:    w={w_nn}, b={b_nn.item():.4f}")

print(f"\nTrue:         w={true_w}, b=3.0")

Closed-form:  w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000
Grad descent: w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000
nn.Linear:    w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000

True:         w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0


In [ ]:
# ✅ SUBMIT
from torch_judge import check
check("linear_regression")